# Explanation


words words words


# §0 - Load pytorch and check version + GPU status 
### (modular)

In [1]:
import pandas as pd
import numpy as np
import torch
print(torch.__version__)
print(torch.cuda.is_available()) 

2.11.0+cpu
False



# §1.1 - Load and clean the data for visualization

## Survival Analysis Preparation
Transforms the Freddie Mac loan-level dataset into a clean DataFrame
ready for Kaplan-Meier, Nelson-Aalen, and Cox Proportional Hazards models
(which can be done using the lifelines library.)

In [2]:
# Import data, source was https://www.kaggle.com/datasets/nikunjhemani/freddie-macs-dataset-pre-processed?resource=download
PATH = "data/Freddie_Mac.csv"
df = pd.read_csv(PATH, index_col=0)
print(f"Loaded: {df.shape[0]} loans with {df.shape[1]} columns each:\n{df.columns}")
df

Loaded: 148938 loans with 35 columns each:
Index(['CreditScore', 'FirstTimeHomebuyer', 'MSA', 'MIP', 'Units', 'Occupancy',
       'OCLTV', 'DTI', 'OrigUPB', 'LTV', 'OrigInterestRate', 'Channel', 'PPM',
       'PropertyState', 'PropertyType', 'LoanPurpose', 'OrigLoanTerm',
       'NumBorrowers', 'SellerName', 'ServicerName', 'EverDelinquent',
       'MonthsDelinquent', 'MonthsInRepayment', 'FirstPayment_Year',
       'FirstPayment_Month', 'Parsed_FirstPaymentDate', 'Maturity_Year',
       'Maturity_Month', 'Parsed_MaturityDate', 'LTV_range', 'Credit_range',
       'YearsInRepayment', 'Repay_range', 'IsFirstTimeHomebuyer', 'Duration'],
      dtype='object')


,CreditScore,FirstTimeHomebuyer,MSA,MIP,Units,Occupancy,OCLTV,DTI,OrigUPB,LTV,...,Parsed_FirstPaymentDate,Maturity_Year,Maturity_Month,Parsed_MaturityDate,LTV_range,Credit_range,YearsInRepayment,Repay_range,IsFirstTimeHomebuyer,Duration
0,670.533671,0,16974,25,1,O,89,27.0,117000,89.0,...,1999-02-01,2029,1,2029-01-01,High,Good,4.333333,2,No,30
1,670.533671,0,19740,0,1,O,73,17.0,109000,73.0,...,1999-02-01,2029,1,2029-01-01,High,Good,12.000000,4,No,30
2,670.533671,0,29940,0,1,O,75,16.0,88000,75.0,...,1999-02-01,2029,1,2029-01-01,High,Good,5.583333,2,No,30
3,670.533671,0,31084,0,1,O,76,14.0,160000,76.0,...,1999-02-01,2029,1,2029-01-01,High,Good,2.916667,1,No,30
4,670.533671,0,35644,0,1,O,78,18.0,109000,78.0,...,1999-02-01,2029,1,2029-01-01,High,Good,4.500000,2,No,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148937,719.000000,1,30660,0,1,O,80,21.0,155000,80.0,...,1999-03-01,2029,2,2029-02-01,High,Good,10.166667,3,Yes,30
148938,719.000000,1,30660,0,1,O,80,41.0,94000,80.0,...,1999-03-01,2029,2,2029-02-01,High,Good,4.583333,2,Yes,30
148939,719.000000,1,30660,0,2,O,70,28.0,111000,70.0,...,1999-03-01,2029,2,2029-02-01,High,Good,13.250000,4,Yes,30
148940,719.000000,1,30660,25,1,O,88,20.0,35000,88.0,...,1999-03-01,2029,2,2029-02-01,High,Good,14.500000,4,Yes,30


In [5]:
# Reduce unneeded dimensions (redundancy)

df = df.drop(columns=[
    # The dataset already contains bucketed/range columns (LTV_range, Credit_range,
    # Repay_range, YearsInRepayment) that were derived from the raw numeric fields.
    # We drop these and work from the raw values so we control the encoding.
    # NOTE: IF NOT DONE we introduce multicollinearity in Cox models.
    "LTV_range",
    "Credit_range",
    "Repay_range",
    "YearsInRepayment",
    "Parsed_FirstPaymentDate",
    "Parsed_MaturityDate",
    # This one we drop just because it is duplicate of FirstTimeHomebuyer (int)
    "IsFirstTimeHomebuyer"   
])
print(f"{df.shape[1]} columns remain")

28 columns remain


In [6]:
# MonthsInRepayment is our TIME AXIS, clean it and perform EDA
df["duration_months"] = df["MonthsInRepayment"].astype(int)

print(f"Number of rows containing impossible months = {sum(np.array(df["duration_months"] < 1))}")

print(f"Duration range: {df['duration_months'].min()} - {df['duration_months'].max()} months")

Number of rows containing impossible months = 0
Duration range: 1 - 212 months


In [7]:
# Like we did with duration_months, our event_observed column is the canonical name for the event indicator in survival analysis
df["event_observed"] = df["EverDelinquent"].astype(int)

print(f"Event breakdown:")
print(df["event_observed"].value_counts().rename({0: "Censored (no event)", 1: "Event (delinquent)"}))
print(f"Event rate: {df['event_observed'].mean():.1%}")
# TODO check THIS!!

Event breakdown:
event_observed
Censored (no event)    107052
Event (delinquent)      41886
Name: count, dtype: int64
Event rate: 28.1%


In [8]:
# Clean covariate columns (all other columns)

# Check for nulls
numeric_cols = ["CreditScore", "LTV", "OCLTV", "DTI", "OrigUPB", "OrigInterestRate",
                "OrigLoanTerm", "NumBorrowers", "MIP", "Units"]
print(f"Null counts in numeric cols:\n{df[numeric_cols].isnull().sum()}")

# Onehot encode categorical columns
cat_cols = ["Channel", "LoanPurpose", "Occupancy", "PropertyType", "PropertyState"]
 
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int) # TODO: CHECK THIS!!
print(f"After encoding categoricals: {df_encoded.shape[1]} columns")
df_encoded.columns

Null counts in numeric cols:
CreditScore         0
LTV                 0
OCLTV               0
DTI                 0
OrigUPB             0
OrigInterestRate    0
OrigLoanTerm        0
NumBorrowers        0
MIP                 0
Units               0
dtype: int64
After encoding categoricals: 41 columns


Index(['CreditScore', 'FirstTimeHomebuyer', 'MSA', 'MIP', 'Units', 'OCLTV',
       'DTI', 'OrigUPB', 'LTV', 'OrigInterestRate', 'PPM', 'OrigLoanTerm',
       'NumBorrowers', 'SellerName', 'ServicerName', 'EverDelinquent',
       'MonthsDelinquent', 'MonthsInRepayment', 'FirstPayment_Year',
       'FirstPayment_Month', 'Maturity_Year', 'Maturity_Month', 'Duration',
       'duration_months', 'event_observed', 'Channel_C', 'Channel_R',
       'Channel_T', 'LoanPurpose_N', 'LoanPurpose_P', 'Occupancy_O',
       'Occupancy_S', 'PropertyType_CP', 'PropertyType_LH', 'PropertyType_MH',
       'PropertyType_PU', 'PropertyType_SF', 'PropertyState_Northeast',
       'PropertyState_Others', 'PropertyState_South',
       'PropertyState_West Coast'],
      dtype='object')

In [9]:
# Extra signals to encode (per Claude)
 
# Rate spread above a rough "market" proxy: how expensive is this loan?
# (Useful because high-rate loans are more likely to refinance/prepay when rates drop)
df_encoded["rate_spread"] = df_encoded["OrigInterestRate"] - df_encoded["OrigInterestRate"].median()
 
# Borrower leverage: combined LTV > 80 is the traditional high-risk threshold
df_encoded["high_ltv"] = (df_encoded["LTV"] > 80).astype(int)
 
# Origination year: captures macro rate environment at origination
df_encoded["orig_year"] = df_encoded["FirstPayment_Year"]
 
# Term flag: 15yr vs 30yr (the dataset is nearly all 30yr but worth flagging)
df_encoded["is_15yr"] = (df_encoded["OrigLoanTerm"] <= 180).astype(int)
 
print("Derived covariates added: rate_spread, high_ltv, orig_year, is_15yr")
 

Derived covariates added: rate_spread, high_ltv, orig_year, is_15yr


In [10]:
# 
drop_for_model = [
    "Unnamed: 0" if "Unnamed: 0" in df_encoded.columns else None,
    "MonthsInRepayment",    # absorbed into duration_months
    "EverDelinquent",       # absorbed into event_observed
    "MonthsDelinquent",     # post-event info would cause data leakage
    "Duration",             # near-constant loan term, not time-to-event
    "FirstPayment_Year",    # absorbed into orig_year
    "FirstPayment_Month",   # ditto
    "Maturity_Year",        # ditto
    "Maturity_Month",       # ditto
    "SellerName",           # uninteresting
    "ServicerName",         # ditto
    "MSA",                  # ditto
]
drop_for_model = [c for c in drop_for_model if c and c in df_encoded.columns]
 
survival_df = df_encoded.drop(columns=drop_for_model)

In [11]:
# Per the lifelines library, we must have duration and event in index 0 and 1 respectively
l0 = "duration_months"
l1 = "event_observed"
cols = list(survival_df.columns)
cols.remove(l0)
cols.remove(l1)
cols.append(l1)
cols.append(l0)
cols = list(reversed(cols))
survival_df = survival_df.reindex(columns=cols)

# NOTE: This index shuffling is LOAD BEARING for later applications, do not forget this step in future builds

print(f"Final survival DataFrame: {survival_df.shape[0]} rows x {survival_df.shape[1]} columns")
print(f"Columns: {list(survival_df.columns)}")

Final survival DataFrame: 148938 rows x 34 columns
Columns: ['duration_months', 'event_observed', 'is_15yr', 'orig_year', 'high_ltv', 'rate_spread', 'PropertyState_West Coast', 'PropertyState_South', 'PropertyState_Others', 'PropertyState_Northeast', 'PropertyType_SF', 'PropertyType_PU', 'PropertyType_MH', 'PropertyType_LH', 'PropertyType_CP', 'Occupancy_S', 'Occupancy_O', 'LoanPurpose_P', 'LoanPurpose_N', 'Channel_T', 'Channel_R', 'Channel_C', 'NumBorrowers', 'OrigLoanTerm', 'PPM', 'OrigInterestRate', 'LTV', 'OrigUPB', 'DTI', 'OCLTV', 'Units', 'MIP', 'FirstTimeHomebuyer', 'CreditScore']


In [12]:
# One row per loan for KM / Cox (no change)
loan_level_df = survival_df.copy()
print(f"Loan-level dataset: {loan_level_df.shape}")
loan_level_df

Loan-level dataset: (148938, 34)


,duration_months,event_observed,is_15yr,orig_year,high_ltv,rate_spread,PropertyState_West Coast,PropertyState_South,PropertyState_Others,PropertyState_Northeast,...,PPM,OrigInterestRate,LTV,OrigUPB,DTI,OCLTV,Units,MIP,FirstTimeHomebuyer,CreditScore
0,52,0,0,1999,1,-0.125,0,0,0,0,...,0,6.750,89.0,117000,27.0,89,1,25,0,670.533671
1,144,0,0,1999,0,-0.375,0,0,1,0,...,0,6.500,73.0,109000,17.0,73,1,0,0,670.533671
2,67,0,0,1999,0,0.000,0,0,1,0,...,0,6.875,75.0,88000,16.0,75,1,0,0,670.533671
3,35,0,0,1999,0,0.000,1,0,0,0,...,0,6.875,76.0,160000,14.0,76,1,0,0,670.533671
4,54,0,0,1999,0,0.250,0,0,0,1,...,0,7.125,78.0,109000,18.0,78,1,0,0,670.533671
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148937,122,0,0,1999,0,0.000,0,0,1,0,...,0,6.875,80.0,155000,21.0,80,1,0,1,719.000000
148938,55,0,0,1999,0,-0.125,0,1,0,0,...,0,6.750,80.0,94000,41.0,80,1,0,1,719.000000
148939,159,0,0,1999,0,-0.625,1,0,0,0,...,1,6.250,70.0,111000,28.0,70,2,0,1,719.000000
148940,174,0,0,1999,1,0.000,0,0,1,0,...,0,6.875,88.0,35000,20.0,88,1,25,1,719.000000


In [13]:
# Expand to loan-month panel for discrete-time hazard models
# TODO NEEDS WORK!!
def expand_to_panel(row):
    """
    Create one row per month for a loan, with event firing only at terminal month.
    """
    months = range(1, int(row["duration_months"]) + 1)
    records = []
    for t in months:
        r = row.copy()
        r["t"] = t
        r["event_this_month"] = 1 if (t == row["duration_months"] and row["event_observed"] == 1) else 0
        records.append(r)
    return records
 
# Note: for very large datasets use a vectorised approach instead of apply
# This sample applies to the first 5,000 loans to demonstrate the structure
sample = survival_df.head(5000)
panel_rows = []
for _, row in sample.iterrows():
    panel_rows.extend(expand_to_panel(row))
 
panel_df = pd.DataFrame(panel_rows)
print(f"Loan-month panel (sample 5k loans): {panel_df.shape}")

Loan-month panel (sample 5k loans): (319550, 36)


In [ ]:
# Save results for use by visualization utilities  
loan_level_df.to_csv("data/survival_loan_level.csv", index=False)
panel_df.to_csv("data/survival_panel_sample.csv", index=False)

# §1.2 - Getting ready for the model

By now the data should be ready for visualization, go to survival_and_hazard_vis.ipynb to check your work. We are now moving on to:

### Temporal train / val / test split

words

In [ ]:
# Checkpoint, does nothing if nb has already been ran top to bottom
LOAN_PATH = "data/survival_loan_level.csv"
PANNEL_PATH = "data/survival_panel_sample.csv"

loan_level_df = pd.read_csv(LOAN_PATH)
panel_df = pd.read_csv(PANNEL_PATH)

In [ ]:
# Hyperparams
HORIZION = 180
# NOTE: For the survival model architecture, I am going to first try a discrete-time hazard network

In [ ]:

# EDA to discover ideal cutoff ranges
print("orig_year distribution:")
print(loan_level_df["orig_year"].value_counts().sort_index())

In [ ]:

# Temporal boundaries — adjust based on the distribution you see above
TRAIN_CUTOFF = 2009   # train: orig_year <= 2009
VAL_CUTOFF   = 2011   # val:   2010–2011   test: 2012+

train_loans = loan_level_df[loan_level_df["orig_year"] <= TRAIN_CUTOFF]
val_loans   = loan_level_df[(loan_level_df["orig_year"] > TRAIN_CUTOFF) &
                             (loan_level_df["orig_year"] <= VAL_CUTOFF)]
test_loans  = loan_level_df[loan_level_df["orig_year"] > VAL_CUTOFF]

print(f"\nTrain: {len(train_loans):,} loans  "
      f"({train_loans['event_observed'].mean():.1%} event rate)")
print(f"Val:   {len(val_loans):,} loans  "
      f"({val_loans['event_observed'].mean():.1%} event rate)")
print(f"Test:  {len(test_loans):,} loans  "
      f"({test_loans['event_observed'].mean():.1%} event rate)")

In [2]:
print("checking kernel")

checking kernel
